# Version 2.2

- (2026.5.14~2026.5.17) Add WQB API script
- (TODO) Fix terminal log display problem 

In [1]:
# Temp install command
# %pip install langchain langchain-text-splitters langchain-chroma langchain-openai langchain-community langchain-huggingface langchain-classic crewai crewai-tools sentence-transformers ipywidgets pypdf tqdm ansi2html

In [1]:
import os
import json
import requests
from pathlib import Path
from crewai import Agent, Task, Crew, Process, LLM
from langchain_chroma import Chroma
from langchain_classic.retrievers import MergerRetriever
from crewai.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from config.api_key import API_KEY_MOONSHOT, API_KEY_GEMINI_C26, API_KEY_GEMINI_CU, API_KEY_DEEPSEEK
import datetime
from utils.logger import setup_logger
from utils.htmlcolorlog import capture_and_log

# ====================== IMPORT FROM YOUR SIMULATOR ======================
from wqb_api.wqb_api_v1 import (
    initialize_global_variables,
    _validate_regular_formula,
    _build_simulation_payload,
    simulate_and_evaluate_alpha
)

### Path Config

In [2]:
# ====================== CONFIG: EVERYTHING ON YOUR OTHER DRIVE ======================
BASE_DIR = Path.cwd() # current directory
# WQB_FORUM_PATH = BASE_DIR / "Docs" / "Forums"
WQB_FORUM_CHINA_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_china_consultant_pdf"
WQB_FORUM_GLOBAL_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_global_consultant_pdf"
WQB_FORUM_RESEARCH_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_research_pdf"
WQB_FORUM_TIPS_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_brain_tips_pdf"
WQB_OFFICIAL_DOCS_PATH = BASE_DIR / "Docs" / "OfficialDocs"
OPERATOR_FILE_PATH = BASE_DIR / "Operators" / "Operators-Agent.json"
DATAFIELDS_FILE_PATH = BASE_DIR / "DataFields" / "Datafield-Dataset-Category-Description.json"
# Note: PaymentPolicy store in WQB_FORUM_TIPS_PATH since it has only few pdfs
# WQB_PAYMENT_POLICY_PATH = BASE_DIR / "Docs" / "PaymentPolicy"

EMBEDDING_DB_FORUM_CHINA_DIR = BASE_DIR / "embedding_db" / "wqb_forum_china_embedding_db"
EMBEDDING_DB_FORUM_GLOBAL_DIR = BASE_DIR / "embedding_db" / "wqb_forum_global_embedding_db"
EMBEDDING_DB_FORUM_RESEARCH_DIR = BASE_DIR / "embedding_db" / "wqb_forum_research_embedding_db"
EMBEDDING_DB_FORUM_TIPS_DIR = BASE_DIR / "embedding_db" / "wqb_forum_tips_embedding_db"
EMBEDDING_DB_OFFICIALDOCS_DIR = BASE_DIR / "embedding_db" / "wqb_official_docs_embedding_db"
EMBEDDING_DB_DIRECTORIES = {
    EMBEDDING_DB_FORUM_CHINA_DIR,
    EMBEDDING_DB_FORUM_GLOBAL_DIR,
    EMBEDDING_DB_FORUM_RESEARCH_DIR,
    EMBEDDING_DB_FORUM_TIPS_DIR,
    EMBEDDING_DB_OFFICIALDOCS_DIR
}
HF_CACHE_DIR = BASE_DIR / "cache" / "hf"
LOG_DIR = BASE_DIR / "logs" / datetime.datetime.now().strftime("%Y%m")

# Create a timestamp for the color html log and transcript files
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
TRANSCRIPT_FILE = LOG_DIR / f"wqb_agent-{timestamp}.transcript.txt"
HTML_FILE = LOG_DIR / f"wqb_agent-{timestamp}.html"

# Create the folders (pathlib makes this easy too)
for directory in [
    HF_CACHE_DIR, EMBEDDING_DB_FORUM_CHINA_DIR, EMBEDDING_DB_FORUM_GLOBAL_DIR, 
    EMBEDDING_DB_FORUM_RESEARCH_DIR, EMBEDDING_DB_FORUM_TIPS_DIR, 
    EMBEDDING_DB_OFFICIALDOCS_DIR, LOG_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# ====================== LOAD JSON DATA ======================
with open(OPERATOR_FILE_PATH, 'r', encoding='utf-8') as f:
    operators_data = json.load(f)

with open(DATAFIELDS_FILE_PATH, 'r', encoding='utf-8') as f:
    datafields_data = json.load(f)

### Log Config

In [3]:
# ====================== INITIALIZE LOGGER ======================
logger = setup_logger(LOG_DIR, "wqb_agent", "shared_logger")

[26-5-20 10:44:40][INFO][SETUP LOG] ✅ logger System Started
[26-5-20 10:44:40][INFO][SETUP LOG] Log file path: d:\AI_Data\Computer\WorldQuantBrain-Agent\logs\202605\wqb_agent-20260520-104440.log


### LLM API

In [4]:
# Get Model List

base_moonshot_url = "https://api.moonshot.cn/v1"
model_moonshot_url = "https://api.moonshot.cn/v1/models"
base_gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
model_gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/models"
base_deepseek_url = "https://api.deepseek.com/v1"
model_deepseek_url = "https://api.deepseek.com/v1/models"
base_url = base_deepseek_url
model_url = model_deepseek_url
# API_KEY = API_KEY_MOONSHOT
API_KEY = API_KEY_DEEPSEEK

In [ ]:
def Get_Model_List(url, api_key):
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        models = response.json()
        logger.info("Main", f"Response: {models}")
        model_list = []
        for model in models['data']:
            model_list.append(model['id'])
            logger.info("Main", f"Model ID: {model['id']}")
        return model_list
    else:
        logger.error("Main", f"Error: {response.status_code}, {response.text}")
        return []

Get_Model_List(model_url, API_KEY)

[26-5-20 10:45:26][INFO][Main] Response: {'object': 'list', 'data': [{'id': 'deepseek-v4-flash', 'object': 'model', 'owned_by': 'deepseek'}, {'id': 'deepseek-v4-pro', 'object': 'model', 'owned_by': 'deepseek'}]}
[26-5-20 10:45:26][INFO][Main] Model ID: deepseek-v4-flash
[26-5-20 10:45:26][INFO][Main] Model ID: deepseek-v4-pro


In [ ]:
# Use your exact proxy settings
# pro_model = "moonshot/kimi-k2.5" # gemini-3.1-pro (if use reserve gemini)
# flash_model = "moonshot/moonshot-v1-128k" # gemini-3.0-flash-thinking (if use reserve gemini)
pro_model = "deepseek/deepseek-v4-pro"
flash_model = "deepseek/deepseek-v4-flash"
# 🚨🚨 CRITICAL WARNING: LLM Provider must be provided. Pass in the LLM provider you are trying to call. Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers

llm_pro = LLM(
    model=pro_model,   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=API_KEY,
    temperature=1,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=3,            # extra retries
)

llm_flash = LLM(
    model=flash_model,   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=API_KEY,
    temperature=0.6,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=3,            # extra retries
)

In [9]:
# LLM query test

def test_llm_query(llm, query):
    try:
        # CHANGE THIS: Use .call() to execute the query
        response = llm.call(query) 
        
        logger.info("LLM Test", f"Query: {query}")
        logger.info("LLM Test", f"Response: {response}")
    except Exception as e:
        logger.error("LLM Test", f"Error: {str(e)}")

test_llm_query(llm_flash, "Who are you and what can you do?")

[26-5-20 10:46:58][INFO][LLM Test] Query: Who are you and what can you do?
[26-5-20 10:46:58][INFO][LLM Test] Response: Hello! I'm DeepSeek, an AI assistant created by the company DeepSeek (深度求索). I'm here to help you with a wide range of tasks!

Here's what I can do:

💬 **Engage in conversation** - Chat about virtually any topic, from casual discussion to deep philosophical questions

📝 **Read and process files** - I can handle images, PDFs, Word documents, Excel files, PowerPoint presentations, and text files - extracting and analyzing the text content

🔗 **Process links** - I can read content from web links you share

🌐 **Search the internet** - When you manually enable the search feature (with the web search button), I can find current information online

📚 **Handle large contexts** - With a 1M token context window, I can process enormous amounts of text - like reading entire book trilogies in one go!

🎯 **Help with tasks** - Writing, analysis, brainstorming, coding, learning, prob

In [10]:
test_llm_query(llm_pro, "Who are you and what can you do?")

[26-5-20 10:47:21][INFO][LLM Test] Query: Who are you and what can you do?
[26-5-20 10:47:21][INFO][LLM Test] Response: Hello! I'm DeepSeek, an AI assistant created by the company DeepSeek (深度求索). I'm here to help you with a wide range of tasks!

Here's what I can do:

**📝 Text & Conversation**
- Answer questions on virtually any topic
- Help with writing, editing, and brainstorming
- Translate between languages
- Summarize long documents
- Explain complex concepts in simple terms

**📄 File Processing**
- Read and analyze uploaded files (images, PDFs, Word docs, Excel sheets, PowerPoint presentations, text files)
- Extract and work with text content from these files

**🔍 Online Search**
- Search the web for real-time information (you need to manually enable this by clicking the search button on the web or app)

**💻 Technical Help**
- Assist with coding and programming questions
- Debug code
- Explain technical concepts

**📱 Voice Input** (on the app)
- Accept voice input on the mobile 

### Embedding (Multiple Database)

- Forums
- Official Docs
- Payment Policy

See `wqbagent_embedding.ipynb`

### Load Embedding Model

In [11]:
# ==================================== Initialize Retriever (for querying) ====================================
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # excellent for Chinese
    cache_folder=str(HF_CACHE_DIR),  # Use the custom cache directory
    model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
    encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
    show_progress=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### Combine multiple embedding database

In [ ]:
# Old single database
# vectorstore = Chroma(persist_directory=EMBEDDING_DB_FORUM_GLOBAL_DIR, embedding_function=embeddings)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# # 2. Initialize a list to hold individual retrievers
# retrievers = []

# for db_dir in EMBEDDING_DB_DIRECTORIES:
#     # Load the vectorstore for each directory
#     vectorstore = Chroma(persist_directory=db_dir, embedding_function=embeddings)
    
#     # Create a retriever for it
#     # Note: If you set k=8 here, each DB will return 8 docs. 
#     # With 5 databases, you'll get up to 40 documents back initially.
#     retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
#     retrievers.append(retriever)

# # 3. Combine all individual retrievers into one
# combined_retriever = MergerRetriever(retrievers=retrievers)

# logger.info("Retriever Initialization", "✅ Combined retriever initialized with multiple embedding databases.")

# Now you can use this exactly like a standard single retriever (test with a simple query)
# combined_retriever.invoke("alpha")

def Combine_Multiple_Embedding_Databases(embedding_db_directories, embeddings):
    retrievers = []
    for db_dir in embedding_db_directories:
        # Load the vectorstore for each directory
        vectorstore = Chroma(persist_directory=db_dir, embedding_function=embeddings)
        # Create a retriever for it
        # Note: If you set k=8 here, each DB will return 8 docs. 
        # With 5 databases, you'll get up to 40 documents back initially.
        retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
        retrievers.append(retriever)
    # 3. Combine all individual retrievers into one
    combined_retriever = MergerRetriever(retrievers=retrievers)
    
    logger.info("Retriever Initialization", "✅ Combined retriever initialized with multiple embedding databases.")
    
    # Now you can use this exactly like a standard single retriever (test with a simple query)
    # combined_retriever.invoke("alpha")
    return combined_retriever

combined_retriever = Combine_Multiple_Embedding_Databases(EMBEDDING_DB_DIRECTORIES, embeddings)

[26-5-20 10:47:48][INFO][Retriever Initialization] ✅ Combined retriever initialized with multiple embedding databases.


### Search Tool

In [13]:
# ====================== DOCS SEARCH TOOL ======================
@tool("retrieve_text_data")
def retrieve_text_data(query: str) -> str:
    """Fetches relevant text snippets based on a string query.
    
    IMPORTANT FORMATTING RULE: 
    The 'query' argument MUST be a plain string. 
    DO NOT pass a dictionary.
    For example, 
    Correct: "momentum" 
    Incorrect: {"type": "str", "value": "momentum"}
    
    Input a highly specific financial concept or math operator (e.g., 'supply chain momentum', 'analyst revision').
    Returns text context to be used for answering user queries."""
    
    docs = combined_retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ====================== JSON SEARCH TOOLS ======================
@tool("search_operators")
def search_operators(query: str) -> str:
    """Search the operator definitions and syntax. 
    Input a concept or specific operator name."""
    results = []
    query_lower = query.lower()
    for op_name, op_details in operators_data.items():
        if (query_lower in op_name.lower() or 
            query_lower in op_details.get('description', '').lower() or 
            query_lower in op_details.get('category', '').lower()):
            
            res = f"Operator: {op_name}\nSyntax: {op_details.get('definition')}\nDesc: {op_details.get('description')}"
            results.append(res)
            
        if len(results) >= 10:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching operators found."

@tool("search_datafields")
def search_datafields(query: str) -> str:
    """Search the data dictionary for dataset fields. 
    Input a concept or specific field name."""
    results = []
    query_lower = query.lower()
    for field_name, field_details in datafields_data.items():
        if (query_lower in field_name.lower() or 
            query_lower in field_details.get('description', '').lower() or 
            query_lower in field_details.get('category_name', '').lower()): # Note: matching your JSON typo 'category_name'
            
            res = f"Field: {field_name}\nType: {field_details.get('type')}\nDesc: {field_details.get('description')}"
            results.append(res)
            
        if len(results) >= 15:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching data fields found."

### Test Search Tools

In [14]:
from wqbquant_searchtool_test import test_search_tools

In [15]:
test_search_tools(retrieve_text_data, "retrieve_text_data", "AllRightsReserved", logger)

[26-5-20 10:47:53][INFO][Test Search Tools] Testing retrieve_text_data tool...


Using Tool: retrieve_text_data


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[26-5-20 10:47:56][INFO][Test Search Tools] 
[26-5-20 10:47:56][INFO][Test Search Tools] TOOL TEST RESULT:
[26-5-20 10:47:56][INFO][Test Search Tools] © 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

unauthorized means of accessing, logging-in or registering on BRAIN.
6. DO NOT use BRAIN in any manner that could interrupt, damage, disable, overburden or impair BRAIN
or interfere with any other party's use and enjoyment of BRAIN
7. DO NOT distribute, publish, and exploit BRAIN or BRAIN Elements unless you have received our
express written prior permission.
8. DO NOT act as representative of BRAIN or WorldQuant without approval from WorldQuant LLC.
9. DO NOT violate the above document or the terms included in the Terms and Conditions.
Failure to follow the above rules would result in severe consequences including termination and
suspension of accounts.
For research best practices

In [16]:
test_search_tools(search_operators, "search_operators", "neutralize", logger)

[26-5-20 10:47:59][INFO][Test Search Tools] Testing search_operators tool...
[26-5-20 10:47:59][INFO][Test Search Tools] 
[26-5-20 10:47:59][INFO][Test Search Tools] TOOL TEST RESULT:
[26-5-20 10:47:59][INFO][Test Search Tools] Operator: bucket
Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False, NaNGroup=False)
or
bucket(rank(x), buckets = “2,5,6,7,10”, skipBoth=False, NaNGroup=False)
Desc: The bucket operator creates custom groups by dividing data into buckets (ranges) based on ranked values of any data field. These buckets can then be used with group operators like group_neutralize, group_rank, group_zscore etc.
---
Operator: group_neutralize
Syntax: group_neutralize(x, group)
Desc: Neutralizes Alpha values within each specified group by subtracting the group mean from each value. Groups can be industry, sector, country, or any custom grouping.
[26-5-20 10:47:59][INFO][Test Search Tools] 
[26-5-20 10:47:59][INFO][Test Search Tools] Length of returned text: 639 characters


Using Tool: search_operators


In [17]:
test_search_tools(search_datafields, "search_datafields", "health", logger)

[26-5-20 10:48:01][INFO][Test Search Tools] Testing search_datafields tool...
[26-5-20 10:48:01][INFO][Test Search Tools] 
[26-5-20 10:48:01][INFO][Test Search Tools] TOOL TEST RESULT:
[26-5-20 10:48:01][INFO][Test Search Tools] Field: community_engagement_score
Type: MATRIX
Desc: Score measuring company contributions to local communities and public health.
---
Field: anl48_bdvd_curr_dvd_health
Type: VECTOR
Desc: BDVD Current DVD Health
---
Field: mdl36_index_rating
Type: MATRIX
Desc: Financial health rating
---
Field: liquidity_coverage_ratio
Type: VECTOR
Desc: Ratio comparing current assets to current liabilities to assess short-term financial health.
---
Field: transport_healthcare_expense_value
Type: VECTOR
Desc: Expenses related to transportation and healthcare.
---
Field: industry_healthcare_services_flag
Type: MATRIX
Desc: Indicator for companies classified in the healthcare services and medical industry.
---
Field: anl11_3pme
Type: MATRIX
Desc: Aggregate KPI for Employee Contin

Using Tool: search_datafields


### Test Search Ability in Agents

> 🚨 Format Error

```json
I encountered an error while trying to use the tool. This was the error: Arguments validation failed: 1 validation error for Retrieve_Text_Data
query
  Input should be a valid string [type=string_type, input_value={'description': None, 'ty...e': 'AllRightsReserved'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type.
 Tool retrieve_text_data accepts these inputs: Tool Name: retrieve_text_data
Tool Arguments: {'query': {'description': None, 'type': 'str'}}
Tool Description: Fetches relevant text snippets based on a string query.
    Input a highly specific financial concept or math operator (e.g., 'supply chain momentum', 'analyst revision').
    Returns text context to be used for answering user queries.
```

This error occurred because the AI agent initially misunderstood how to format its input for the `retrieve_text_data` tool.

Under the hood, CrewAI uses a library called Pydantic to strictly validate the inputs sent to your tools. Here is the exact breakdown of why that validation failed:

#### The Mismatch

* **What the tool expected:** The Python function was defined as `def retrieve_text_data(query: str) -> str:`. This tells the system that the `query` argument must be a simple, raw string.
* *Correct format:* `{"query": "AllRightsReserved"}`


* **What the agent actually sent:** The LLM got confused by the tool's JSON schema and tried to pass an entire dictionary containing metadata instead of just the value.
* *Incorrect format:* `{"query": {"description": null, "type": "str", "value": "AllRightsReserved"}}`


#### The Result

Because Pydantic was expecting a `string` but received a `dict` (dictionary), it immediately threw the `type=string_type` validation error and blocked the execution:

> *Input should be a valid string [...] input_type=dict*

#### How it got resolved

To stop this error from happening in the first place—and to save the time and tokens your agent wastes on retrying—you need to give the underlying LLM an unambiguous map of exactly what data to send.

While the agent’s self-correction is great, relying on it is inefficient. Here are the two best solutions to fix this permanently in CrewAI/LangChain:

- 👍 Solution : Explicit Docstring Engineering (The Quick Fix)

If you don't want to import Pydantic models, you can sometimes brute-force the LLM into compliance by being aggressively explicit in the tool's docstring. The docstring is injected directly into the LLM's prompt, so adding a formatting warning can do the trick.

```python
@tool("retrieve_text_data")
def retrieve_text_data(query: str) -> str:
    """Fetches relevant text snippets based on a string query.
    
    IMPORTANT FORMATTING RULE: 
    The 'query' argument MUST be a plain string. 
    DO NOT pass a dictionary. 
    Correct: "AllRightsReserved" 
    Incorrect: {"type": "str", "value": "AllRightsReserved"}
    
    Input a highly specific financial concept or math operator.
    Returns text context to be used for answering user queries."""
    
    docs = combined_retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

```

In [18]:
from wqbquant_searchtool_test import test_agents
test_agents(
    retrieve_text_data=retrieve_text_data, 
    search_operators=search_operators, 
    search_datafields=search_datafields, 
    llm=llm_flash, 
    transcript_file=TRANSCRIPT_FILE,
    html_file=HTML_FILE,
    logger=logger
)

[26-5-20 10:48:24][INFO][Test Agents] 🚀 Kickstarting Crew process.


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 946aec98-66a0-4006-896a-5192e653023c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: API Formatting Assistant                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Perform a strict health check on the system's tools.                                                       │
│                                                                                                                 │
│      STEP 1: Execute a test for each available search function:                                                 │
│      - search 'AllRightsReserved' using `retrieve_text_data`                                                    │
│      - search 'neutralize' using `search_operators`                                                             │
│      - search 'health' using `search_datafields`                                                                │
│                                                                                                                 │
│      STEP 2: Only AFTER you have observed the results from all three tools, formulate your final response.      │
│      If any tool fails, returns an error, or is denied, clearly state the failure in the final report.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6d894649-e437-4e2e-8cff-b07a3e1c9808                                                                     │
│  Agent: API Formatting Assistant                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 946aec98-66a0-4006-896a-5192e653023c                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ## System Tools Health Check Report                                                              │
│                                                                                                                 │
│  - **retrieve_text_data**: ✅PASS - "© 2026 WorldQuant BRAIN®. All Rights Reserved."                            │
│  - **search_operators**: ✅PASS - "Operator: bucket Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False,  │
│  NaNGroup=False)"                                                                                               │
│  - **search_datafields**: ✅PASS - "Field: community_engagement_score Type: MATRIX Desc: Score measuring        │
│  company contributions to local communities and public health."                                                 │
│                                                                                                                 │
│  All three tools responded successfully without errors, denials, or failures.                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[26-5-20 10:49:41][INFO][Test Agents] ✅ Crew kickoff completed successfully.
[26-5-20 10:49:41][INFO][Test Agents] 
FINAL RESULT
## System Tools Health Check Report

- **retrieve_text_data**: ✅PASS - "© 2026 WorldQuant BRAIN®. All Rights Reserved."
- **search_operators**: ✅PASS - "Operator: bucket Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False, NaNGroup=False)"
- **search_datafields**: ✅PASS - "Field: community_engagement_score Type: MATRIX Desc: Score measuring company contributions to local communities and public health."

All three tools responded successfully without errors, denials, or failures.



[INFO] 🌐 Colorful HTML Log saved to: d:\AI_Data\Computer\WorldQuantBrain-Agent\logs\202605\wqb_agent-20260520-104437.html


CrewOutput(raw='## System Tools Health Check Report\n\n- **retrieve_text_data**: ✅PASS - "© 2026 WorldQuant BRAIN®. All Rights Reserved."\n- **search_operators**: ✅PASS - "Operator: bucket Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False, NaNGroup=False)"\n- **search_datafields**: ✅PASS - "Field: community_engagement_score Type: MATRIX Desc: Score measuring company contributions to local communities and public health."\n\nAll three tools responded successfully without errors, denials, or failures.', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description="\n    Perform a strict health check on the system's tools.\n\n    STEP 1: Execute a test for each available search function:\n    - search 'AllRightsReserved' using `retrieve_text_data`\n    - search 'neutralize' using `search_operators`\n    - search 'health' using `search_datafields`\n\n    STEP 2: Only AFTER you have observed the results from all three tools, formulate your final response.\n    If any tool fail

### WQB API Tool

In [19]:
# Initialize simulator global variables (operators, datafields, multipliers)
# This must be called before the agents attempt to use the tools
account_no = "0"  # You can change this if needed
success, init_msg = initialize_global_variables(account_no=account_no)
if not success: raise RuntimeError(f"Failed to initialize simulator: {init_msg}")

[26-5-20 10:50:02][INFO][Get Datafield-Dataset (0)] ✅ Datafield-Dataset dict loaded successfully from local file.
[26-5-20 10:50:02][INFO][Get DatasetID Suffix (0)] ✅ DatasetID suffix-category dict loaded successfully from local file.
[26-5-20 10:50:02][INFO][Get Operators (0)] ✅ Operators loaded successfully from local file.
[26-5-20 10:50:05][INFO][Get Pyramid Multipliers (0)] ✅ Pyramid multipliers loaded successfully from local file.
[26-5-20 10:50:05][INFO][Initialize Global Variables (0)] ✅ Global variables initialized successfully.


In [20]:
@tool("check_regular_formula")
def check_regular_formula(regular_formula: str, region: str, delay: int, universe: str) -> str:
    """
    Validates the syntax and data fields of the regular formula locally BEFORE running a full simulation.
    Call this to detect invalid datafields, bad syntax, or missing operators.
    
    Inputs:
    - regular_formula: The alpha mathematical expression.
    - region: The region {"USA", "GLB", "EUR", "ASI", "CHN", "IND", "KOR", "TWN", "MEA"}
    - delay: Integer delay, 0 or 1. The rule: {
        "USA": {0, 1},
        "GLB": {1},
        "EUR": {0, 1},
        "ASI": {1},
        "CHN": {0, 1},
        "IND": {1},
        "KOR": {1},
        "TWN": {1},
        "MEA": {1}
    }
    - universe: The target universe. The rule: {
        "USA": {"TOP3000", "TOP2000", "TOP1000", "TOP500", "TOP200", "TOPSP500", "ILLIQUID_MINVOL1M"},
        "GLB": {"TOP3000", "MINVOL1M", "MINVOL10M", "TOPDIV3000"},
        "EUR": {"TOP2500", "TOP1200", "TOP800", "TOP400", "ILLIQUID_MINVOL1M", "TOPCS1600"},
        "ASI": {"TOP500", "MINVOL1M", "MINVOL10M", "ILLIQUID_MINVOL1M"},
        "CHN": {"TOP2000U"},
        "IND": {"TOP500"},
        "KOR": {"TOP600"},
        "TWN": {"TOP500", "TOP100"},
        "MEA": {"TOP400", "TOP300"}
    }
    """
    try:
        success, result = _validate_regular_formula(
            regular_formula=regular_formula,
            region=region,
            delay=int(delay),
            universe=universe,
            account_no=account_no # global variable set during initialization
        )
        if not success:
            return f"❌ Validation Failed: {result}\nReview the datafields and operators used."
        return f"✅ Formula Check Passed! Multiplier details: {result}"
    except Exception as e:
        return f"❌ Checker Exception: {e}"

@tool("wqb_simulate_api")
def wqb_simulate_api(settings: str, regular_formula: str) -> str:
    """
    Run full alpha simulation and evaluation by calling the WQB API.
    
    Inputs:
    - settings: JSON string of simulator settings. Must included keys: [
        "instrumentType", # default "EQUITY"
        "region", # "USA", "GLB", "EUR", "ASI", "CHN", "IND", "KOR", "TWN", "MEA"
        "universe", # depends on region, e.g. "TOP3000", "MINVOL1M", etc.
        "delay", # 0 or 1, depends on region
        "decay", # between 0 and 512, suggest to be integer for better performance, but float is also acceptable
        "neutralization", # depends on region, e.g. "SECTOR", "INDUSTRY", "COUNTRY", "GICS_SECTOR"
        "truncation", # between 0 and 1, inclusive, float only, suggest keep 2 decimal places for better performance, but more decimal places are also acceptable
        "pasteurization", # either ON or OFF
        "nanHandling", # either ON or OFF
        "testPeriod", # Format 1: with year and without month, like "P1Y"; Format 2: with year and month, like "P1Y2M"; Format 3: with month only, like "P2M"
        "maxTrade", # either ON or OFF
        "maxPosition" # either ON or OFF
        # Max Position and Max Trade cannot both be set to On simultaneously
    ]
    - regular_formula: The WorldQuant expression string.
    
    Output:
    Returns the JSON payload containing Simulation Status, IS_Checks, and Correlation.
    If the status is not 'COMPLETE' or IS_Checks fail, read the feedback and try again!
    """
    try:
        settings_payload = json.loads(settings) if isinstance(settings, str) else settings
    except json.JSONDecodeError as e:
        return f"❌ Invalid settings JSON: {e}"

    # 1. Build Payload and validate settings exactly as the simulator expects
    success, sim_payload = _build_simulation_payload(settings_payload, regular_formula, account_no="0")
    if not success:
        return f"❌ Payload Build Error: {sim_payload}"

    # 2. Run simulation and evaluation
    success, result = simulate_and_evaluate_alpha(
        alpha_settings=sim_payload,
        regular=regular_formula,
        account_no=account_no, # global variable set during initialization
        include_self_corr=True,
        include_prod_corr=True
    )
    
    if not success:
        return f"❌ API Request Failed: {result}"
        
    return json.dumps(result, ensure_ascii=False, indent=2)

### Agents & Crew

In [ ]:
# ====================== AGENTS (Your Quant Research Team) ======================
# 💡 Note: # allow_delegation=False means the agent cannot delegate to other agents and must complete the task itself. 
# This is important for the researcher to ensure it fully utilizes the retrieval tool and doesn't skip steps.
researcher = Agent(
    role="WorldQuant Docs Researcher & Master Analyst",
    goal="Use the `retrieve_text_data` tool to fetch context for the user's request. Base your output on the tool's results. Never answer from general knowledge.",
    backstory="""You are a veteran WorldQuant Brain consultant. You are an advanced AI agent equipped with a local vector database interface.
    You do not have the PDFs in your internal memory; you rely ENTIRELY on the `retrieve_text_data` tool. You use lateral thinking. 
    If a user asks about 'volume', you search for 'liquidity shock', 'turnover spike', or 'institutional block trades'.
    You always call the `retrieve_text_data` tool multiple times with completely different vocabulary each time to get a full picture.
    """,
    tools=[retrieve_text_data],
    llm=llm_pro,
    verbose=True,
    allow_delegation=False
)

ideator = Agent(
    role="Low-Correlation BRAIN Innovator",
    goal="Create truly innovative, submittable alphas using specific alternative data fields.",
    backstory="""You are a contrarian quant. You MUST use the `search_datafields` tool to find real, specific dataset names (like 'anti_pollution_policy_industry_rank') to build your hypotheses. Do not hallucinate data field names.""",
    tools=[search_datafields], # <--- ADDED TOOL
    llm=llm_pro,
    verbose=True
)

coder = Agent(
    role="WorldQuant BRAIN Expression Expert",
    goal="Convert the idea into a valid expression using exact Operator syntax and Exact Data fields.",
    backstory="""You are an ex-WorldQuant Brain coder. 
    1. You MUST use `search_operators` to check the exact syntax of functions (e.g., checking if add() takes a filter argument). 
    2. You MUST use `search_datafields` to ensure you are using the exact string name for data sets.
    You never guess operator syntax.""",
    tools=[search_operators, search_datafields], # <--- ADDED TOOLS
    llm=llm_flash,
    verbose=True
)

# validator = Agent(
#     role="WorldQuant Submission Validator",
#     goal="Ensure the alpha is innovative, low-correlation, and ready to submit. Output ONLY in the exact user-specified format.",
#     backstory="You are the final gatekeeper. You check for simulator compatibility, low correlation, and economic soundness. You never add extra text outside the required format.",
#     tools=[wqb_simulate_api],
#     llm=llm_pro,
#     verbose=True
# )

# New: Validator can now call the simulator tool to get real feedback and iteratively improve the formula until it passes IS checks with a COMPLETE status.
validator = Agent(
    role="WorldQuant Submission Validator & Iterative Optimizer",
    goal="Ensure the alpha simulates correctly, passes all IS checks, and is ready to submit. Output ONLY in the exact user-specified format.",
    backstory="""You are the final gatekeeper and debugging expert. You never pass a broken alpha. 
    You are highly skilled at reading WQB API error messages (like 'Unknown Operator' or 'Invalid Datafield') 
    and iteratively tweaking the formula until the API returns a 'COMPLETE' status with passing IS criteria.
    You never give up on the first error; you adjust and resimulate.""",
    tools=[check_regular_formula, wqb_simulate_api],  # Equipped with the real tools
    llm=llm_pro,
    verbose=True,
    allow_delegation=False
)

# ====================== TASKS & CREW ======================
task1 = Task(
    description="""
    Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.
    
    STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.
    STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor, call it again with a different phrase.
    STEP 3: Synthesize the returned database snippets.
    
    Focus on extracting real opinions and specific discussions.
    """,
    expected_output="Structured summary of the retrieved database content with direct quotes.",
    agent=researcher
)

task2 = Task(
    description="""
    Generate 3-5 genuinely innovative alpha ideas.
    CRITICAL: You MUST use the `search_datafields` tool to search for keywords related to your ideas (e.g., "ESG", "Analyst", "Supply Chain") and include the EXACT field names in your output.
    Focus on low correlation and economic rationale.
    """,
    expected_output="Numbered list of 3-5 alpha ideas containing exact Data Field names, hypothesis, and low-correlation justification.",
    agent=ideator
)

task3 = Task(
    description="""
    Take the BEST idea from Task 2 and write a clean, valid WorldQuant BRAIN expression.
    CRITICAL: You MUST use the `search_operators` tool to verify the syntax of every math/logic function you plan to use before writing the final expression.
    Choose realistic Target Settings (Region, Universe, Neutralization, Delay, Decay, Truncation).
    """,
    expected_output="One complete alpha in the exact user format (Alpha Name + Economic Hypothesis + Target Settings + Full BRAIN Expression).",
    agent=coder
)

# task4 = Task(
#     description="""
#     Act as strict WorldQuant reviewer.
#     Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic sense.
#     If needed, improve it slightly.

#     CRITICAL: Before finalizing, call `wqb_simulate_api` exactly once with:
#     - settings: a JSON string that includes Region, Universe, Neutralization, Delay, Decay, and Truncation.
#     - regular_formula: the final formula string.

#     Use simulation output to decide whether the alpha is good enough; if not, improve the formula once and return the improved one.

#     THEN output ONLY the final alpha in the EXACT format the user wants:
    
#     **Alpha Name:** ...
#     **Economic Hypothesis:** ...
#     **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ | Truncation: ___
#     **Full BRAIN Expression:** ...
    
#     Do not add any extra explanation or text outside this format.
#     """,
#     expected_output="Final alpha in the exact markdown format requested by the user.",
#     agent=validator
# )

# New: Validator now has an iterative workflow to debug and improve the formula until it passes the simulator checks with a COMPLETE status.
task4 = Task(
    description="""
    Act as a strict WorldQuant reviewer and iterate until the alpha is perfect.
    
    CRITICAL WORKFLOW:
    1. Validate the formula locally using the `check_regular_formula` tool. Pass the formula, region, delay, and universe.
       - If it fails, fix the datafields or operators and check again.
    2. Once local validation passes, call `wqb_simulate_api` with the full settings JSON and the formula.
       - Wait for the API response. 
    3. Analyze the API Output:
       - Look at `"simulation": {"status": ... }`. If it is "ERROR", read the message, modify the formula, and re-run step 2.
       - Look at `"evaluation": {"is_checks": {"Status": ... }}`. If it's "FAIL" (e.g., low Sharpe, high Turnover), tweak your formula parameters, operators, or settings, and re-run step 2.
    4. Repeat this iterative debugging process up to 4 times until you achieve a 'COMPLETE' status and a 'PASS' in IS_Checks.

    THEN output ONLY the final working alpha in the EXACT format the user wants:
    
    **Alpha Name:** ...
    **Economic Hypothesis:** ...
    **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ | Truncation: ___
    **Full BRAIN Expression:** ...
    
    Do not add any extra explanation, reasoning, or debugging text outside this format in your final output.
    """,
    expected_output="Final working alpha in the exact markdown format requested by the user.",
    agent=validator
)

crew = Crew(
    agents=[researcher, ideator, coder, validator],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,
    verbose=True,
    max_rpm=12
    # tracing=True
)

### How to reduce the rate limit of llm api request?

If the rate-limiting is happening on the LLM side (DeepSeek or Moonshot), managing how the agents talk to the language models is exactly what you need.

Here is how `rpm_max` works under the hood in CrewAI to save your LLM API quotas.

- How `rpm_max` Works in CrewAI

`rpm_max` stands for **Requests Per Minute Maximum**. It acts as a traffic cop (a rate limiter) sitting between your CrewAI agents and the LLM API provider (DeepSeek/Moonshot).

When you set `rpm_max` on the Crew or an Agent, CrewAI sets up a global timer wrapper around every single call to `llm.call()` or `llm.generate()`.

```
[Agent Thought] ──> [ CrewAI Rate Limiter ] ──> (Delay if too fast) ──> [ DeepSeek API ]

```

1. Mathematical Spacing (The Token Bucket Principle)

CrewAI doesn't just let the agents make 20 requests in 2 seconds and then block them for the remaining 58 seconds. Instead, it calculates the minimum safe interval between requests.

If you set `rpm_max=20`, CrewAI calculates:


$$\text{Interval} = \frac{60 \text{ seconds}}{20 \text{ requests}} = 3 \text{ seconds per request}$$

If an agent finishes a thought and tries to make another LLM call only $1$ second after the last one, CrewAI will **automatically force a 2-second sleep** before sending the payload to DeepSeek.

1. Handling the "Thought-Action-Observation" Loop

In frameworks like CrewAI, an agent doesn't just call the LLM once per task. It uses an **ReAct (Reasoning and Acting)** loop:

1. **Thought:** LLM decides what to do (Request 1).
2. **Action:** Agent calls a tool (e.g., `search_operators`).
3. **Observation:** Tool returns data.
4. **Thought:** LLM reads tool data and decides next steps (Request 2).

Without `rpm_max`, this loop happens at lightning speed. If `search_operators` returns instantly, Request 1 and Request 2 might hit DeepSeek within $200\text{ms}$ of each other. DeepSeek sees this sudden burst from the same API key and flags it as a DDoS or rate-limit abuse, throwing a `429 Error`. `rpm_max` flattens these spikes out.

- Where to Put It in Your Code

You have two choices for where to apply this configuration, depending on how strict you want to be.

Option A: Global Crew Level (Easiest)

This sets a blanket rule for the entire execution. If one agent is waiting, the whole system paces itself.

```python
crew = Crew(
    agents=[researcher, ideator, coder, validator],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,
    verbose=True,
    rpm_max=15,  # 💡 Enforces a ~4 second delay between ANY LLM request across the crew
)

```

Option B: Specific Agent Level (Targeted)

If your `researcher` and `ideator` are behaving fine, but your `validator` or `coder` are looping like crazy and hitting the API too fast, you can target just them:

```python
validator = Agent(
    role="WorldQuant Submission Validator & Iterative Optimizer",
    goal="Ensure the alpha simulates correctly...",
    backstory="...",
    tools=[check_regular_formula, wqb_simulate_api],
    llm=llm_pro,
    verbose=True,
    allow_delegation=False,
    rpm_max=10  # 💡 Only this agent will be forced to wait 6 seconds between thoughts
)

```

- One Final DeepSeek Tip

DeepSeek-V4 models are incredibly popular and experience severe server-side congestion. If you are using their official endpoint (`api.deepseek.com`), they enforce strict limits on **Concurrent Requests** (how many requests are processing at the exact same millisecond).

Setting `rpm_max` to something conservative like `10` or `15` is the most effective way to prevent your local agent framework from overwhelming their endpoints.

In [ ]:
# ====================== RUN ======================
if __name__ == "__main__":
    result = crew.kickoff(inputs={
        "user_request": """
        Explore forum discussions specifically around Analyst Estimates (EPS, Revisions) or Supply Chain inventory data. 
        Find out what basic combinations people are using, and generate alphas that apply non-linear operators 
        (like sign, abs, or conditional logic) to these specific datasets to achieve low correlation.
        """.strip()
    })
    logger.info("\n\n=== FINAL RESULT ===")
    logger.info(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 550f0102-65e7-453c-9e4f-adff6b0d4315                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: WorldQuant Docs Researcher & Master Analyst                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.         │
│                                                                                                                 │
│      STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.                    │
│      STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor,  │
│  call it again with a different phrase.                                                                         │
│      STEP 3: Synthesize the returned database snippets.                                                         │
│                                                                                                                 │
│      Focus on extracting real opinions and specific discussions.                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: WorldQuant Submission Validator & Iterative Optimizer                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "Thought": "First, validate the formula locally using check_regular_formula.",                               │
│    "Action": "check_regular_formula",                                                                           │
│    "Action Input": {                                                                                            │
│      "regular_formula": "trade_when(vwap >= ts_mean(vwap, 20), rank(ts_rank(close, 252)), -1)",                 │
│      "region": "USA",                                                                                           │
│      "delay": 1,                                                                                                │
│      "universe": "TOP3000"                                                                                      │
│    }                                                                                                            │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: c5f88806-53d2-4ef3-8ca1-b45dde5f7ccd                                                                     │
│  Agent: WorldQuant Submission Validator & Iterative Optimizer                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 550f0102-65e7-453c-9e4f-adff6b0d4315                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "Thought": "First, validate the formula locally using check_regular_formula.",                               │
│    "Action": "check_regular_formula",                                                                           │
│    "Action Input": {                                                                                            │
│      "regular_formula": "trade_when(vwap >= ts_mean(vwap, 20), rank(ts_rank(close, 252)), -1)",                 │
│      "region": "USA",                                                                                           │
│      "delay": 1,                                                                                                │
│      "universe": "TOP3000"                                                                                      │
│    }                                                                                                            │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL RESULT ===
```json
{
  "Thought": "First, validate the formula locally using check_regular_formula.",
  "Action": "check_regular_formula",
  "Action Input": {
    "regular_formula": "trade_when(vwap >= ts_mean(vwap, 20), rank(ts_rank(close, 252)), -1)",
    "region": "USA",
    "delay": 1,
    "universe": "TOP3000"
  }
}
```
